In [ ]:
# 单元格1：导入必要的库和设置基础路径
import h5py
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm  # 进度条显示
import datetime
import glob

# 设置基础路径
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data'
data_path = os.path.join(base_dir, 'DATA/TRAIN38.mat')
output_base_dir = os.path.join(base_dir, 'processed_data')

# 创建输出目录
os.makedirs(output_base_dir, exist_ok=True)

print(f"数据路径: {data_path}")
print(f"输出目录: {output_base_dir}")

In [ ]:
# 单元格2：加载数据并进行初步分析
# 加载TRAIN38.mat文件
f = h5py.File(data_path, 'r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

# 提取数据
data = arrays['data'].transpose()  # 转置以获取正确的形状
region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()

# 数据基本信息
print(f"数据形状: {data.shape}")
print(f"标签形状: {region.shape}")
print(f"病人索引形状: {prob_idx.shape}")

# 分析唯一的病人ID和标签值
unique_prob_idx = np.unique(prob_idx)
print(f"唯一病人ID: {unique_prob_idx}")
print(f"病人总数: {len(unique_prob_idx)}个")

# 分析标签情况
if region.ndim == 2:
    # 如果标签是多列的
    for i in range(region.shape[1]):
        unique_values = np.unique(region[:, i])
        print(f"标签列 {i} 的唯一值: {unique_values}, 类别数: {len(unique_values)}")
else:
    # 如果标签是单列的
    unique_values = np.unique(region)
    print(f"标签的唯一值: {unique_values}, 类别数: {len(unique_values)}")

# 分析每个病人的样本数量
for idx in unique_prob_idx:
    count = np.sum(prob_idx == idx)
    print(f"病人 {idx}: {count}个样本")

In [ ]:
# 单元格3（修正版）：按照病人ID划分数据集
# 功能：确保voxel、region和prob_idx的一一对应关系

# 设置随机种子确保结果可重现
np.random.seed(42)

# 病人划分
test_patients = [38]  # 第38号病人直接进入测试集
remaining_patients = [i for i in range(1, 38)]  # 剩余37个病人

# 随机选择7个病人加入测试集
additional_test_patients = np.random.choice(remaining_patients, 7, replace=False)
test_patients.extend(additional_test_patients)

# 剩余的30个病人用于训练和验证
train_val_patients = [p for p in remaining_patients if p not in additional_test_patients]

print(f"测试集病人 ({len(test_patients)}个): {sorted(test_patients)}")
print(f"训练和验证集病人 ({len(train_val_patients)}个): {sorted(train_val_patients)}")

# 根据病人ID划分数据
test_indices = np.where(np.isin(prob_idx, test_patients))[0]
train_val_indices = np.where(np.isin(prob_idx, train_val_patients))[0]

# 提取测试集 - 确保索引一致性
test_data = data[test_indices]
test_regions = region[test_indices]
test_prob_idx = prob_idx[test_indices]

# 提取训练和验证集 - 确保索引一致性
train_val_data = data[train_val_indices]
train_val_regions = region[train_val_indices]
train_val_prob_idx = prob_idx[train_val_indices]

print(f"测试集样本数: {len(test_data)}")
print(f"训练和验证集样本数: {len(train_val_data)}")

# 验证一一对应关系
print("\n验证数据分割后的一一对应关系:")
print(f"测试集: 数据形状 {test_data.shape}, 标签形状 {test_regions.shape}, 病人ID形状 {test_prob_idx.shape}")
print(f"训练和验证集: 数据形状 {train_val_data.shape}, 标签形状 {train_val_regions.shape}, 病人ID形状 {train_val_prob_idx.shape}")

# 保存病人分配信息
patient_allocation = {
    "test_patients": sorted(test_patients),
    "train_val_patients": sorted(train_val_patients)
}

# 释放原始完整数据的内存
del data, region, prob_idx, arrays

In [ ]:
# 单元格4（全面修正版）：按标签分组处理数据
# 功能：将训练和验证数据按标签分组，随机打乱，并按6:2比例拆分
# 重点：确保voxel、region和prob_idx完全一一对应

# 创建输出目录
train_dir = os.path.join(output_base_dir, 'train')
val_dir = os.path.join(output_base_dir, 'val')
test_dir = os.path.join(output_base_dir, 'test')

for directory in [train_dir, val_dir, test_dir]:
    os.makedirs(directory, exist_ok=True)

# 处理标签
label_data = {}  # 按标签存储体素数据
label_regions = {}  # 按标签存储完整标签数据
label_prob_idx = {}  # 按标签存储病人ID信息

# 判断标签是一维还是多维
if train_val_regions.ndim == 1:
    # 标签是一维的
    unique_labels = np.unique(train_val_regions)
    
    for label in unique_labels:
        # 找到该标签的所有样本 - 重要:使用相同的indices确保一一对应
        indices = np.where(train_val_regions == label)[0]
        samples = train_val_data[indices]
        regions = train_val_regions[indices]  # 保存完整标签信息
        patient_ids = train_val_prob_idx[indices]  # 保存对应的病人ID
        
        # 验证一一对应关系
        assert len(samples) == len(regions) == len(patient_ids), f"标签 {label} 的数据、标签和病人ID长度不一致!"
        
        # 随机打乱 - 关键是使用相同的随机索引打乱所有相关数据
        shuffle_indices = np.random.permutation(len(samples))
        samples = samples[shuffle_indices]
        regions = regions[shuffle_indices]  # 使用相同的索引打乱标签
        patient_ids = patient_ids[shuffle_indices]  # 使用相同的索引打乱病人ID
        
        # 按6:2比例拆分
        train_size = int(len(samples) * 0.6)
        train_samples = samples[:train_size]
        train_regions = regions[:train_size]
        train_patient_ids = patient_ids[:train_size]
        val_samples = samples[train_size:]
        val_regions = regions[train_size:]
        val_patient_ids = patient_ids[train_size:]
        
        # 再次验证一一对应关系
        assert len(train_samples) == len(train_regions) == len(train_patient_ids), f"训练集: 标签 {label} 的数据、标签和病人ID长度不一致!"
        assert len(val_samples) == len(val_regions) == len(val_patient_ids), f"验证集: 标签 {label} 的数据、标签和病人ID长度不一致!"
        
        # 存储
        label_data[int(label)] = {
            "train": train_samples,
            "val": val_samples
        }
        label_regions[int(label)] = {
            "train": train_regions,
            "val": val_regions
        }
        label_prob_idx[int(label)] = {
            "train": train_patient_ids,
            "val": val_patient_ids
        }
        
        print(f"标签 {label}: 总样本 {len(samples)}，训练集 {len(train_samples)}，验证集 {len(val_samples)}")
        
        # 验证完整性
        print(f"  - 训练集: 数据 {train_samples.shape}, 标签 {train_regions.shape}, 病人ID {train_patient_ids.shape}")
        print(f"  - 验证集: 数据 {val_samples.shape}, 标签 {val_regions.shape}, 病人ID {val_patient_ids.shape}")
else:
    # 标签是多维的
    # 找出有效的标签列（包含多个值的列）
    valid_columns = []
    for i in range(train_val_regions.shape[1]):
        if len(np.unique(train_val_regions[:, i])) > 1:
            valid_columns.append(i)
    
    print(f"有效标签列: {valid_columns}")
    
    # 确定使用哪一列作为主标签
    # 这里假设第一个有效列是主标签
    main_label_col = valid_columns[0] if valid_columns else 0
    print(f"使用列 {main_label_col} 作为主标签")
    
    # 按主标签分组
    unique_labels = np.unique(train_val_regions[:, main_label_col])
    
    for label in unique_labels:
        # 找到该标签的所有样本 - 使用相同的indices确保一一对应
        indices = np.where(train_val_regions[:, main_label_col] == label)[0]
        samples = train_val_data[indices]
        regions = train_val_regions[indices]  # 保存完整标签信息
        patient_ids = train_val_prob_idx[indices]  # 保存对应的病人ID
        
        # 验证一一对应关系
        assert len(samples) == len(regions) == len(patient_ids), f"标签 {label} 的数据、标签和病人ID长度不一致!"
        
        # 随机打乱 - 使用相同的随机索引打乱所有相关数据
        shuffle_indices = np.random.permutation(len(samples))
        samples = samples[shuffle_indices]
        regions = regions[shuffle_indices]  # 使用相同的索引打乱标签
        patient_ids = patient_ids[shuffle_indices]  # 使用相同的索引打乱病人ID
        
        # 按6:2比例拆分
        train_size = int(len(samples) * 0.6)
        train_samples = samples[:train_size]
        train_regions = regions[:train_size]
        train_patient_ids = patient_ids[:train_size]
        val_samples = samples[train_size:]
        val_regions = regions[train_size:]
        val_patient_ids = patient_ids[train_size:]
        
        # 再次验证一一对应关系
        assert len(train_samples) == len(train_regions) == len(train_patient_ids), f"训练集: 标签 {label} 的数据、标签和病人ID长度不一致!"
        assert len(val_samples) == len(val_regions) == len(val_patient_ids), f"验证集: 标签 {label} 的数据、标签和病人ID长度不一致!"
        
        # 存储
        label_data[int(label)] = {
            "train": train_samples,
            "val": val_samples
        }
        label_regions[int(label)] = {
            "train": train_regions,
            "val": val_regions
        }
        label_prob_idx[int(label)] = {
            "train": train_patient_ids,
            "val": val_patient_ids
        }
        
        print(f"标签 {label}: 总样本 {len(samples)}，训练集 {len(train_samples)}，验证集 {len(val_samples)}")
        
        # 验证完整性
        print(f"  - 训练集: 数据 {train_samples.shape}, 标签 {train_regions.shape}, 病人ID {train_patient_ids.shape}")
        print(f"  - 验证集: 数据 {val_samples.shape}, 标签 {val_regions.shape}, 病人ID {val_patient_ids.shape}")

# 全局验证数据完整性
print("\n验证数据完整性:")
for label in label_data:
    for split in ["train", "val"]:
        samples = label_data[label][split]
        regions = label_regions[label][split]
        patient_ids = label_prob_idx[label][split]
        
        if len(samples) != len(regions) or len(samples) != len(patient_ids):
            print(f"警告: 标签 {label} 的 {split} 集中，样本、标签和病人ID数量不匹配!")
        else:
            print(f"标签 {label} 的 {split} 集数据完整性验证通过 ({len(samples)} 个样本)")

print("数据分组和拆分完成！")

In [ ]:
# 单元格5：创建和保存StandardScaler
# 功能：使用所有训练数据拟合StandardScaler并保存

# 合并所有训练样本
all_train_samples = []
for label, data_dict in label_data.items():
    all_train_samples.append(data_dict["train"])

all_train_samples = np.vstack(all_train_samples)
print(f"用于拟合Scaler的训练样本总数: {len(all_train_samples)}")

# 拟合StandardScaler
scaler = StandardScaler()
scaler.fit(all_train_samples)

# 保存Scaler
scaler_path = os.path.join(output_base_dir, 'data_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"Scaler已拟合并保存到: {scaler_path}")
print(f"Scaler均值形状: {scaler.mean_.shape}")
print(f"Scaler方差形状: {scaler.var_.shape}")

# 释放内存
del all_train_samples

In [ ]:
# 单元格6（全面修正版）：处理训练集和验证集数据
# 功能：对训练集和验证集数据应用标准化，并按标签保存
# 重点：同时保存完整的region和prob_idx数据，确保三者一一对应

# 处理训练集
print("处理训练集...")
for label, data_dict in tqdm(label_data.items()):
    train_samples = data_dict["train"]
    train_regions = label_regions[label]["train"]
    train_prob_ids = label_prob_idx[label]["train"]
    
    # 验证一一对应
    assert len(train_samples) == len(train_regions) == len(train_prob_ids), f"训练集标签 {label} 数据不一致!"
    
    # 标准化
    train_samples_scaled = scaler.transform(train_samples)
    
    # 创建基础文件名
    base_filename = f"label_{label}_samples_{len(train_samples)}"
    
    # 保存数据
    voxel_file = os.path.join(train_dir, f"{base_filename}_voxels.npy")
    region_file = os.path.join(train_dir, f"{base_filename}_regions.npy")
    prob_idx_file = os.path.join(train_dir, f"{base_filename}_prob_idx.npy")
    
    np.save(voxel_file, train_samples_scaled)
    np.save(region_file, train_regions)
    np.save(prob_idx_file, train_prob_ids)
    
    print(f"保存标签 {label} 的 {len(train_samples)} 个训练样本:")
    print(f"  - 体素数据: {voxel_file}")
    print(f"  - 标签数据: {region_file}")
    print(f"  - 病人ID: {prob_idx_file}")

# 处理验证集
print("处理验证集...")
for label, data_dict in tqdm(label_data.items()):
    val_samples = data_dict["val"]
    val_regions = label_regions[label]["val"]
    val_prob_ids = label_prob_idx[label]["val"]
    
    # 验证一一对应
    assert len(val_samples) == len(val_regions) == len(val_prob_ids), f"验证集标签 {label} 数据不一致!"
    
    # 标准化
    val_samples_scaled = scaler.transform(val_samples)
    
    # 创建基础文件名
    base_filename = f"label_{label}_samples_{len(val_samples)}"
    
    # 保存数据
    voxel_file = os.path.join(val_dir, f"{base_filename}_voxels.npy")
    region_file = os.path.join(val_dir, f"{base_filename}_regions.npy")
    prob_idx_file = os.path.join(val_dir, f"{base_filename}_prob_idx.npy")
    
    np.save(voxel_file, val_samples_scaled)
    np.save(region_file, val_regions)
    np.save(prob_idx_file, val_prob_ids)
    
    print(f"保存标签 {label} 的 {len(val_samples)} 个验证样本:")
    print(f"  - 体素数据: {voxel_file}")
    print(f"  - 标签数据: {region_file}")
    print(f"  - 病人ID: {prob_idx_file}")

print("训练集和验证集处理完成！")

# 释放内存
del label_data, label_regions, label_prob_idx

In [ ]:
# 单元格7（全面修正版）：处理测试集数据
# 功能：对测试集数据应用标准化，并按标签和病人保存
# 重点：确保voxel、region和prob_idx三者一一对应

# 判断标签是一维还是多维
if test_regions.ndim == 1:
    # 标签是一维的
    unique_labels = np.unique(test_regions)
    
    for label in tqdm(unique_labels, desc="处理测试集"):
        # 找到该标签的所有样本 - 使用相同的索引确保一一对应
        indices = np.where(test_regions == label)[0]
        samples = test_data[indices]
        regions = test_regions[indices]
        p_idx = test_prob_idx[indices]
        
        # 验证一一对应
        assert len(samples) == len(regions) == len(p_idx), f"测试集标签 {label} 数据不一致!"
        
        # 标准化
        samples_scaled = scaler.transform(samples)
        
        # 创建基础文件名
        base_filename = f"label_{label}_samples_{len(samples)}"
        
        # 保存全部数据
        voxel_file = os.path.join(test_dir, f"{base_filename}_voxels.npy")
        region_file = os.path.join(test_dir, f"{base_filename}_regions.npy")
        prob_idx_file = os.path.join(test_dir, f"{base_filename}_prob_idx.npy")
        
        np.save(voxel_file, samples_scaled)
        np.save(region_file, regions)
        np.save(prob_idx_file, p_idx)
        
        print(f"保存标签 {label} 的 {len(samples)} 个测试样本:")
        print(f"  - 体素数据: {voxel_file}")
        print(f"  - 标签数据: {region_file}")
        print(f"  - 病人ID: {prob_idx_file}")
        
        # 按病人分别保存（可选）
        for p_id in np.unique(p_idx):
            # 使用相同的索引确保一一对应
            p_indices = np.where(p_idx == p_id)[0]
            p_samples = samples_scaled[p_indices]
            p_regions = regions[p_indices]
            p_idx_data = p_idx[p_indices]
            
            # 验证一一对应
            assert len(p_samples) == len(p_regions) == len(p_idx_data), f"测试集病人 {p_id} 标签 {label} 数据不一致!"
            
            if len(p_samples) > 0:
                p_base_filename = f"patient_{p_id}_label_{label}_samples_{len(p_samples)}"
                p_voxel_file = os.path.join(test_dir, f"{p_base_filename}_voxels.npy")
                p_region_file = os.path.join(test_dir, f"{p_base_filename}_regions.npy")
                p_prob_idx_file = os.path.join(test_dir, f"{p_base_filename}_prob_idx.npy")
                
                np.save(p_voxel_file, p_samples)
                np.save(p_region_file, p_regions)
                np.save(p_prob_idx_file, p_idx_data)
else:
    # 标签是多维的
    # 使用与训练集相同的主标签列
    main_label_col = 0  # 使用单元格4中确定的主标签列
    unique_labels = np.unique(test_regions[:, main_label_col])
    
    for label in tqdm(unique_labels, desc="处理测试集"):
        # 找到该标签的所有样本 - 使用相同的索引确保一一对应
        indices = np.where(test_regions[:, main_label_col] == label)[0]
        samples = test_data[indices]
        regions = test_regions[indices]
        p_idx = test_prob_idx[indices]
        
        # 验证一一对应
        assert len(samples) == len(regions) == len(p_idx), f"测试集标签 {label} 数据不一致!"
        
        # 标准化
        samples_scaled = scaler.transform(samples)
        
        # 创建基础文件名
        base_filename = f"label_{label}_samples_{len(samples)}"
        
        # 保存全部数据
        voxel_file = os.path.join(test_dir, f"{base_filename}_voxels.npy")
        region_file = os.path.join(test_dir, f"{base_filename}_regions.npy")
        prob_idx_file = os.path.join(test_dir, f"{base_filename}_prob_idx.npy")
        
        np.save(voxel_file, samples_scaled)
        np.save(region_file, regions)
        np.save(prob_idx_file, p_idx)
        
        print(f"保存标签 {label} 的 {len(samples)} 个测试样本:")
        print(f"  - 体素数据: {voxel_file}")
        print(f"  - 标签数据: {region_file}")
        print(f"  - 病人ID: {prob_idx_file}")
        
        # 按病人分别保存（可选）
        for p_id in np.unique(p_idx):
            # 使用相同的索引确保一一对应
            p_indices = np.where(p_idx == p_id)[0]
            p_samples = samples_scaled[p_indices]
            p_regions = regions[p_indices]
            p_idx_data = p_idx[p_indices]
            
            # 验证一一对应
            assert len(p_samples) == len(p_regions) == len(p_idx_data), f"测试集病人 {p_id} 标签 {label} 数据不一致!"
            
            if len(p_samples) > 0:
                p_base_filename = f"patient_{p_id}_label_{label}_samples_{len(p_samples)}"
                p_voxel_file = os.path.join(test_dir, f"{p_base_filename}_voxels.npy")
                p_region_file = os.path.join(test_dir, f"{p_base_filename}_regions.npy")
                p_prob_idx_file = os.path.join(test_dir, f"{p_base_filename}_prob_idx.npy")
                
                np.save(p_voxel_file, p_samples)
                np.save(p_region_file, p_regions)
                np.save(p_prob_idx_file, p_idx_data)

print("测试集处理完成！")

# 释放内存
del test_data, test_regions, test_prob_idx

In [ ]:
# 单元格8（全面修正版）：创建数据集索引文件
# 功能：为每个数据集创建索引文件，包含完整的文件引用

def create_dataset_index(directory):
    """为指定目录创建数据集索引文件，包含完整的文件引用"""
    index_file = os.path.join(directory, "label_index.txt")
    
    with open(index_file, 'w') as f:
        f.write("label_id,voxel_count,voxel_file,region_file,prob_idx_file\n")
        
        # 获取所有包含'voxels.npy'的文件，但不包括按病人分类的文件
        data_files = [file for file in os.listdir(directory) 
                     if file.endswith('voxels.npy') and not file.startswith("patient_")]
        
        for file in sorted(data_files, key=lambda x: int(x.split('_')[1])):
            # 从文件名提取信息
            parts = file.split('_')
            label_id = parts[1]
            voxel_count = parts[3]
            
            # 构建对应的region和prob_idx文件名
            region_file = file.replace('voxels.npy', 'regions.npy')
            prob_idx_file = file.replace('voxels.npy', 'prob_idx.npy')
            
            # 验证文件是否存在
            if not os.path.exists(os.path.join(directory, region_file)):
                region_file = "N/A"
            if not os.path.exists(os.path.join(directory, prob_idx_file)):
                prob_idx_file = "N/A"
            
            f.write(f"{label_id},{voxel_count},{file},{region_file},{prob_idx_file}\n")
    
    print(f"索引文件已创建: {index_file}")
    
    # 如果有按病人分类的文件，也为它们创建索引
    patient_files = [file for file in os.listdir(directory) 
                    if file.endswith('voxels.npy') and file.startswith("patient_")]
    
    if patient_files:
        patient_index_file = os.path.join(directory, "patient_index.txt")
        
        with open(patient_index_file, 'w') as f:
            f.write("patient_id,label_id,voxel_count,voxel_file,region_file,prob_idx_file\n")
            
            for file in sorted(patient_files, key=lambda x: (int(x.split('_')[1]), int(x.split('_')[3]))):
                # 从文件名提取信息
                parts = file.split('_')
                patient_id = parts[1]
                label_id = parts[3]
                voxel_count = parts[5]
                
                # 构建对应的region和prob_idx文件名
                region_file = file.replace('voxels.npy', 'regions.npy')
                prob_idx_file = file.replace('voxels.npy', 'prob_idx.npy')
                
                # 验证文件是否存在
                if not os.path.exists(os.path.join(directory, region_file)):
                    region_file = "N/A"
                if not os.path.exists(os.path.join(directory, prob_idx_file)):
                    prob_idx_file = "N/A"
                
                f.write(f"{patient_id},{label_id},{voxel_count},{file},{region_file},{prob_idx_file}\n")
        
        print(f"病人索引文件已创建: {patient_index_file}")

# 为每个数据集创建索引
print("创建数据集索引文件...")
create_dataset_index(train_dir)
create_dataset_index(val_dir)
create_dataset_index(test_dir)

In [ ]:
# 单元格9：创建处理汇总信息文件
# 功能：记录数据处理的详细信息，便于查阅和复现

# 计算各数据集样本总数
def count_samples(directory):
    """计算目录中的样本总数"""
    total = 0
    for file in os.listdir(directory):
        if file.endswith('.npy') and "samples_" in file and not file.startswith("patient_"):
            parts = file.split('_')
            count = int(parts[3])
            total += count
    return total

train_samples = count_samples(train_dir)
val_samples = count_samples(val_dir)
test_samples = count_samples(test_dir)

# 创建汇总信息文件
summary_file = os.path.join(output_base_dir, "processing_summary.txt")

with open(summary_file, 'w') as f:
    f.write("脑体素数据处理汇总信息\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("1. 数据来源\n")
    f.write(f"- 原始数据文件: {data_path}\n\n")
    
    f.write("2. 数据集划分\n")
    f.write(f"- 测试集病人 ({len(patient_allocation['test_patients'])}个): {patient_allocation['test_patients']}\n")
    f.write(f"- 训练和验证集病人 ({len(patient_allocation['train_val_patients'])}个): {patient_allocation['train_val_patients']}\n\n")
    
    f.write("3. 数据集统计\n")
    f.write(f"- 训练集: {train_samples} 个样本\n")
    f.write(f"- 验证集: {val_samples} 个样本\n")
    f.write(f"- 测试集: {test_samples} 个样本\n")
    f.write(f"- 总样本: {train_samples + val_samples + test_samples} 个样本\n\n")
    
    f.write("4. 标准化信息\n")
    f.write(f"- Scaler文件: {os.path.basename(scaler_path)}\n")
    f.write(f"- 特征数量: {len(scaler.mean_)}\n\n")
    
    f.write("5. 标签信息\n")
    f.write(f"- 训练集标签文件: {os.path.join('train', 'label_index.txt')}\n")
    f.write(f"- 验证集标签文件: {os.path.join('val', 'label_index.txt')}\n")
    f.write(f"- 测试集标签文件: {os.path.join('test', 'label_index.txt')}\n\n")
    
    f.write("6. 处理时间\n")
    f.write(f"- 处理完成时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"处理汇总信息已保存到: {summary_file}")

In [ ]:
# 单元格10：创建Scaler使用示例代码
# 功能：提供如何使用保存的Scaler进行数据标准化的示例代码

example_code_file = os.path.join(output_base_dir, "scaler_usage_example.py")

with open(example_code_file, 'w') as f:
    f.write("""# 示例：如何使用保存的Scaler处理新数据
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

def load_scaler(scaler_path):
    \"\"\"加载保存的StandardScaler\"\"\"
    with open(scaler_path, 'rb') as f:
        return pickle.load(f)

def standardize_data(data, scaler):
    \"\"\"使用加载的Scaler标准化数据\"\"\"
    return scaler.transform(data)

if __name__ == "__main__":
    # 1. 加载Scaler
    scaler_path = "data_scaler.pkl"  # 替换为实际路径
    scaler = load_scaler(scaler_path)
    
    # 2. 加载需要标准化的数据
    # 示例：从文件加载数据
    # data = np.load("your_data_file.npy")
    # 或者从其他来源获取数据
    data = np.random.rand(100, 341)  # 示例数据，实际使用时替换为真实数据
    
    # 3. 使用Scaler进行标准化
    standardized_data = standardize_data(data, scaler)
    
    # 4. 使用标准化后的数据进行预测或其他操作
    # model.predict(standardized_data)
    
    # 打印信息
    print("数据标准化完成！")
    print(f"原始数据形状: {data.shape}")
    print(f"标准化后数据形状: {standardized_data.shape}")
""")

print(f"Scaler使用示例代码已保存到: {example_code_file}")

In [ ]:
# 单元格11（全面修正版）：数据加载辅助函数
# 功能：提供加载处理后数据的辅助函数，确保三者一一对应

data_loader_file = os.path.join(output_base_dir, "data_loader.py")

with open(data_loader_file, 'w') as f:
    f.write("""# 数据加载辅助函数
import numpy as np
import os
import glob
import pickle

def load_index(index_file):
    \"\"\"加载标签索引文件\"\"\"
    index_data = {}
    with open(index_file, 'r') as f:
        # 跳过标题行
        header = next(f).strip().split(',')
        has_region = 'region_file' in header
        has_prob_idx = 'prob_idx_file' in header
        
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                label_id = int(parts[0])
                voxel_count = int(parts[1])
                voxel_file = parts[2]
                
                file_info = {
                    'count': voxel_count, 
                    'voxel_file': voxel_file
                }
                
                # 添加region文件信息
                if has_region and len(parts) > 3:
                    region_file = parts[3]
                    file_info['region_file'] = region_file if region_file != 'N/A' else None
                
                # 添加prob_idx文件信息
                if has_prob_idx and len(parts) > 4:
                    prob_idx_file = parts[4]
                    file_info['prob_idx_file'] = prob_idx_file if prob_idx_file != 'N/A' else None
                
                index_data[label_id] = file_info
    return index_data

def load_data_by_label(data_dir, label_id=None, include_metadata=True):
    \"\"\"
    加载指定目录下的数据，可选择按标签过滤
    
    参数:
        data_dir: 数据目录路径
        label_id: 要加载的标签ID，如果为None则加载所有标签
        include_metadata: 是否同时加载标签和病人ID信息
    
    返回:
        如果include_metadata为False:
            data: 加载的体素数据
            labels: 对应的标签ID
        如果include_metadata为True:
            data: 加载的体素数据
            labels: 对应的标签ID
            regions: 完整的标签数据
            prob_idx: 病人ID信息
    \"\"\"
    index_file = os.path.join(data_dir, "label_index.txt")
    index_data = load_index(index_file)
    
    data_list = []
    labels_list = []
    regions_list = []
    prob_idx_list = []
    
    if label_id is not None:
        # 只加载指定标签
        if label_id in index_data:
            info = index_data[label_id]
            voxel_file = info['voxel_file']
            voxel_path = os.path.join(data_dir, voxel_file)
            
            if os.path.exists(voxel_path):
                data = np.load(voxel_path)
                labels = np.full(data.shape[0], label_id)
                
                data_list.append(data)
                labels_list.append(labels)
                
                # 加载标签数据（如果有）
                if include_metadata and 'region_file' in info and info['region_file']:
                    region_path = os.path.join(data_dir, info['region_file'])
                    if os.path.exists(region_path):
                        regions = np.load(region_path)
                        regions_list.append(regions)
                    else:
                        # 如果找不到region文件，使用标签ID填充
                        regions_list.append(labels)
                elif include_metadata:
                    # 如果没有region信息，使用标签ID填充
                    regions_list.append(labels)
                
                # 加载病人ID信息（如果有）
                if include_metadata and 'prob_idx_file' in info and info['prob_idx_file']:
                    prob_idx_path = os.path.join(data_dir, info['prob_idx_file'])
                    if os.path.exists(prob_idx_path):
                        prob_idx = np.load(prob_idx_path)
                        prob_idx_list.append(prob_idx)
                    else:
                        # 如果找不到prob_idx文件，使用零填充
                        prob_idx_list.append(np.zeros(data.shape[0]))
                elif include_metadata:
                    # 如果没有prob_idx信息，使用零填充
                    prob_idx_list.append(np.zeros(data.shape[0]))
    else:
        # 加载所有标签
        for label_id, info in index_data.items():
            voxel_file = info['voxel_file']
            voxel_path = os.path.join(data_dir, voxel_file)
            
            if os.path.exists(voxel_path):
                data = np.load(voxel_path)
                labels = np.full(data.shape[0], label_id)
                
                data_list.append(data)
                labels_list.append(labels)
                
                # 加载标签数据（如果有）
                if include_metadata and 'region_file' in info and info['region_file']:
                    region_path = os.path.join(data_dir, info['region_file'])
                    if os.path.exists(region_path):
                        regions = np.load(region_path)
                        regions_list.append(regions)
                    else:
                        # 如果找不到region文件，使用标签ID填充
                        regions_list.append(labels)
                elif include_metadata:
                    # 如果没有region信息，使用标签ID填充
                    regions_list.append(labels)
                
                # 加载病人ID信息（如果有）
                if include_metadata and 'prob_idx_file' in info and info['prob_idx_file']:
                    prob_idx_path = os.path.join(data_dir, info['prob_idx_file'])
                    if os.path.exists(prob_idx_path):
                        prob_idx = np.load(prob_idx_path)
                        prob_idx_list.append(prob_idx)
                    else:
                        # 如果找不到prob_idx文件，使用零填充
                        prob_idx_list.append(np.zeros(data.shape[0]))
                elif include_metadata:
                    # 如果没有prob_idx信息，使用零填充
                    prob_idx_list.append(np.zeros(data.shape[0]))
    
    if not data_list:
        if include_metadata:
            return np.array([]), np.array([]), np.array([]), np.array([])
        else:
            return np.array([]), np.array([])
    
    if include_metadata:
        # 验证数据一致性
        all_data = np.vstack(data_list)
        all_labels = np.concatenate(labels_list)
        all_regions = np.concatenate(regions_list)
        all_prob_idx = np.concatenate(prob_idx_list)
        
        assert len(all_data) == len(all_labels) == len(all_regions) == len(all_prob_idx), "加载的数据、标签和病人ID长度不一致!"
        
        return all_data, all_labels, all_regions, all_prob_idx
    else:
        return np.vstack(data_list), np.concatenate(labels_list)

def load_all_datasets(base_dir, include_metadata=True):
    \"\"\"
    加载所有数据集
    
    参数:
        base_dir: 基础目录路径，包含train、val、test子目录
        include_metadata: 是否同时加载标签和病人ID信息
    
    返回:
        一个字典，包含训练集、验证集和测试集的数据和标签
    \"\"\"
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'val')
    test_dir = os.path.join(base_dir, 'test')
    
    result = {}
    
    if include_metadata:
        train_data, train_labels, train_regions, train_prob_idx = load_data_by_label(train_dir, include_metadata=True)
        val_data, val_labels, val_regions, val_prob_idx = load_data_by_label(val_dir, include_metadata=True)
        test_data, test_labels, test_regions, test_prob_idx = load_data_by_label(test_dir, include_metadata=True)
        
        result = {
            'train': {'data': train_data, 'labels': train_labels, 'regions': train_regions, 'prob_idx': train_prob_idx},
            'val': {'data': val_data, 'labels': val_labels, 'regions': val_regions, 'prob_idx': val_prob_idx},
            'test': {'data': test_data, 'labels': test_labels, 'regions': test_regions, 'prob_idx': test_prob_idx}
        }
    else:
        train_data, train_labels = load_data_by_label(train_dir, include_metadata=False)
        val_data, val_labels = load_data_by_label(val_dir, include_metadata=False)
        test_data, test_labels = load_data_by_label(test_dir, include_metadata=False)
        
        result = {
            'train': {'data': train_data, 'labels': train_labels},
            'val': {'data': val_data, 'labels': val_labels},
            'test': {'data': test_data, 'labels': test_labels}
        }
    
    return result

def load_scaler(base_dir):
    \"\"\"加载保存的StandardScaler\"\"\"
    scaler_path = os.path.join(base_dir, 'data_scaler.pkl')
    with open(scaler_path, 'rb') as f:
        return pickle.load(f)

def load_patient_data(data_dir, patient_id, label_id=None, include_metadata=True):
    \"\"\"
    加载特定病人的数据
    
    参数:
        data_dir: 数据目录路径
        patient_id: 病人ID
        label_id: 标签ID（可选，如果只想加载特定标签的数据）
        include_metadata: 是否同时加载标签和病人ID信息
    
    返回:
        如果include_metadata为False:
            data: 该病人的体素数据
            labels: 对应的标签ID
        如果include_metadata为True:
            data: 该病人的体素数据
            labels: 对应的标签ID
            regions: 完整的标签数据
            prob_idx: 病人ID信息
    \"\"\"
    patient_data = []
    patient_labels = []
    patient_regions = []
    patient_prob_idx = []
    
    # 查找该病人的所有文件
    pattern = f"patient_{patient_id}_*.npy"
    if label_id is not None:
        pattern = f"patient_{patient_id}_label_{label_id}_*.npy"
        
    patient_files = glob.glob(os.path.join(data_dir, pattern))
    voxel_files = [f for f in patient_files if f.endswith('voxels.npy')]
    
    for voxel_file in voxel_files:
        # 加载体素数据
        data = np.load(voxel_file)
        
        # 从文件名提取标签
        file_name = os.path.basename(voxel_file)
        parts = file_name.split('_')
        try:
            current_label = int(parts[3])  # 假设标签是文件名中的第四个部分
            
            patient_data.append(data)
            patient_labels.append(np.full(data.shape[0], current_label))
            
            # 加载标签数据（如果有）
            if include_metadata:
                region_file = voxel_file.replace('voxels.npy', 'regions.npy')
                if os.path.exists(region_file):
                    regions = np.load(region_file)
                    patient_regions.append(regions)
                else:
                    # 如果找不到region文件，使用标签ID填充
                    patient_regions.append(np.full(data.shape[0], current_label))
                
                # 加载病人ID信息（如果有）
                prob_idx_file = voxel_file.replace('voxels.npy', 'prob_idx.npy')
                if os.path.exists(prob_idx_file):
                    prob_idx = np.load(prob_idx_file)
                    patient_prob_idx.append(prob_idx)
                else:
                    # 如果找不到prob_idx文件，使用病人ID填充
                    patient_prob_idx.append(np.full(data.shape[0], patient_id))
        except:
            print(f"警告: 无法从文件名 {file_name} 中提取标签信息")
    
    if not patient_data:
        if include_metadata:
            return np.array([]), np.array([]), np.array([]), np.array([])
        else:
            return np.array([]), np.array([])
    
    if include_metadata:
        # 验证数据一致性
        all_data = np.vstack(patient_data)
        all_labels = np.concatenate(patient_labels)
        all_regions = np.concatenate(patient_regions)
        all_prob_idx = np.concatenate(patient_prob_idx)
        
        assert len(all_data) == len(all_labels) == len(all_regions) == len(all_prob_idx), "加载的数据、标签和病人ID长度不一致!"
        
        return all_data, all_labels, all_regions, all_prob_idx
    else:
        return np.vstack(patient_data), np.concatenate(patient_labels)

def verify_data_consistency(base_dir):
    \"\"\"
    验证数据集的一致性，确保所有文件正确对应
    
    参数:
        base_dir: 基础目录路径，包含train、val、test子目录
    
    返回:
        检查结果（布尔值）和问题列表
    \"\"\"
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'val')
    test_dir = os.path.join(base_dir, 'test')
    
    issues = []
    
    for directory in [train_dir, val_dir, test_dir]:
        voxel_files = [f for f in os.listdir(directory) if f.endswith('voxels.npy')]
        
        for voxel_file in voxel_files:
            region_file = voxel_file.replace('voxels.npy', 'regions.npy')
            prob_idx_file = voxel_file.replace('voxels.npy', 'prob_idx.npy')
            
            voxel_path = os.path.join(directory, voxel_file)
            region_path = os.path.join(directory, region_file)
            prob_idx_path = os.path.join(directory, prob_idx_file)
            
            # 检查文件是否存在
            if not os.path.exists(region_path):
                issues.append(f"缺少文件: {region_path}")
                continue
                
            if not os.path.exists(prob_idx_path):
                issues.append(f"缺少文件: {prob_idx_path}")
                continue
            
            # 检查文件长度是否一致
            try:
                voxel_data = np.load(voxel_path)
                region_data = np.load(region_path)
                prob_idx_data = np.load(prob_idx_path)
                
                if len(voxel_data) != len(region_data) or len(voxel_data) != len(prob_idx_data):
                    issues.append(f"数据长度不一致: {voxel_file} ({len(voxel_data)}), {region_file} ({len(region_data)}), {prob_idx_file} ({len(prob_idx_data)})")
            except Exception as e:
                issues.append(f"加载文件出错: {voxel_file}, {str(e)}")
    
    return len(issues) == 0, issues

# 使用示例
if __name__ == "__main__":
    # 替换为实际路径
    base_dir = "processed_data"
    
    # 验证数据一致性
    print("验证数据集一致性...")
    success, issues = verify_data_consistency(base_dir)
    if success:
        print("所有数据集通过一致性验证!")
    else:
        print(f"发现 {len(issues)} 个问题:")
        for issue in issues:
            print(f"  - {issue}")
    
    # 加载所有数据集，包含完整元数据
    print("\\n加载数据集...")
    datasets = load_all_datasets(base_dir, include_metadata=True)
    
    # 打印数据集信息
    for dataset_name, dataset in datasets.items():
        print(f"{dataset_name}集: {dataset['data'].shape[0]} 个样本, {len(np.unique(dataset['labels']))} 个唯一标签")
        print(f"  - 数据形状: {dataset['data'].shape}")
        print(f"  - 标签形状: {dataset['labels'].shape}")
        print(f"  - 完整标签形状: {dataset['regions'].shape}")
        print(f"  - 病人ID形状: {dataset['prob_idx'].shape}")
        print(f"  - 包含 {len(np.unique(dataset['prob_idx']))} 个唯一病人ID")
    
    # 加载Scaler
    scaler = load_scaler(base_dir)
    print(f"\\nScaler已加载，特征数量: {len(scaler.mean_)}")
    
    # 加载特定病人的数据
    test_dir = os.path.join(base_dir, 'test')
    patient_id = 38  # 替换为实际病人ID
    patient_data, patient_labels, patient_regions, patient_prob_idx = load_patient_data(test_dir, patient_id)
    if len(patient_data) > 0:
        print(f"\\n病人 {patient_id} 的数据: {patient_data.shape[0]} 个样本, {len(np.unique(patient_labels))} 个唯一标签")
        print(f"  - 数据形状: {patient_data.shape}")
        print(f"  - 标签形状: {patient_labels.shape}")
        print(f"  - 完整标签形状: {patient_regions.shape}")
        print(f"  - 病人ID形状: {patient_prob_idx.shape}")
        
        # 验证这些数据是否来自该病人
        if np.all(patient_prob_idx == patient_id):
            print(f"  - 验证通过：所有数据确实来自病人 {patient_id}")
        else:
            print(f"  - 验证失败：数据中包含其他病人的信息")
""")

print(f"数据加载辅助函数已保存到: {data_loader_file}")

In [ ]:
# 单元格12（全面修正版）：验证处理结果
# 功能：全面验证处理后的数据集合是否一一对应

print("验证处理结果...")

# 验证输出目录结构
print("1. 验证目录结构:")
for directory in [train_dir, val_dir, test_dir]:
    voxel_files = [f for f in os.listdir(directory) if f.endswith('voxels.npy')]
    region_files = [f for f in os.listdir(directory) if f.endswith('regions.npy')]
    prob_idx_files = [f for f in os.listdir(directory) if f.endswith('prob_idx.npy')]
    
    print(f"  - {os.path.basename(directory)}目录:")
    print(f"    * {len(voxel_files)} 个体素数据文件")
    print(f"    * {len(region_files)} 个标签数据文件")
    print(f"    * {len(prob_idx_files)} 个病人ID数据文件")

# 验证三者的一一对应关系
print("\n2. 验证数据一一对应关系:")
for directory in [train_dir, val_dir, test_dir]:
    issues = []
    
    voxel_files = [f for f in os.listdir(directory) if f.endswith('voxels.npy')]
    
    for voxel_file in voxel_files:
        # 构造对应的region和prob_idx文件名
        base_name = voxel_file[:-10]  # 去掉'voxels.npy'
        region_file = f"{base_name}regions.npy"
        prob_idx_file = f"{base_name}prob_idx.npy"
        
        # 检查文件是否存在
        if not os.path.exists(os.path.join(directory, region_file)):
            issues.append(f"标签文件缺失: {region_file}")
        if not os.path.exists(os.path.join(directory, prob_idx_file)):
            issues.append(f"病人ID文件缺失: {prob_idx_file}")
            
        # 如果文件都存在，检查长度是否一致
        if (os.path.exists(os.path.join(directory, region_file)) and 
            os.path.exists(os.path.join(directory, prob_idx_file))):
            
            voxel_data = np.load(os.path.join(directory, voxel_file))
            region_data = np.load(os.path.join(directory, region_file))
            prob_idx_data = np.load(os.path.join(directory, prob_idx_file))
            
            if len(voxel_data) != len(region_data):
                issues.append(f"体素数据({len(voxel_data)})和标签数据({len(region_data)})长度不一致: {voxel_file}")
            if len(voxel_data) != len(prob_idx_data):
                issues.append(f"体素数据({len(voxel_data)})和病人ID数据({len(prob_idx_data)})长度不一致: {voxel_file}")
    
    if issues:
        print(f"  - {os.path.basename(directory)}目录中发现 {len(issues)} 个问题:")
        for issue in issues:
            print(f"    * {issue}")
    else:
        print(f"  - {os.path.basename(directory)}目录数据一一对应关系验证通过!")

# 验证索引文件
print("\n3. 验证索引文件:")
for directory in [train_dir, val_dir, test_dir]:
    index_file = os.path.join(directory, "label_index.txt")
    if os.path.exists(index_file):
        with open(index_file, 'r') as f:
            headers = next(f).strip().split(',')
            line_count = sum(1 for _ in f)
        print(f"  - {os.path.basename(directory)}索引文件: {line_count} 个条目")
        print(f"    * 索引字段: {headers}")

# 验证Scaler
print("\n4. 验证Scaler:")
try:
    with open(scaler_path, 'rb') as f:
        loaded_scaler = pickle.load(f)
    
    print(f"  - Scaler加载成功, 特征数量: {len(loaded_scaler.mean_)}")
    
    # 从训练数据中获取一些样本用于验证
    train_files = glob.glob(os.path.join(train_dir, "*voxels.npy"))
    if train_files:
        test_data = np.load(train_files[0])[:10]  # 只取10个样本用于测试
        transformed1 = scaler.transform(test_data)
        transformed2 = loaded_scaler.transform(test_data)
        
        is_equal = np.allclose(transformed1, transformed2)
        print(f"  - Scaler验证: {'成功' if is_equal else '失败'}")
except Exception as e:
    print(f"  - Scaler验证出错: {str(e)}")

print("\n处理完成！所有数据已按要求处理并保存，并保持了数据三者的一一对应关系。")

In [ ]:
# 单元格13：总结和下一步建议
print("""
================================================================================
                           数据处理完成总结
================================================================================

处理流程概述:
1. 从TRAIN38.mat文件中加载数据
2. 将病人分配到测试集和训练+验证集
3. 将训练+验证集数据按标签分组并随机打乱
4. 按6:2比例将训练+验证集拆分为训练集和验证集
5. 使用训练集数据拟合StandardScaler并保存
6. 对所有数据集应用相同的标准化处理
7. 将处理后的数据按标签分别保存
8. 创建索引文件和辅助函数

输出文件:
- 训练集数据: {}/train/*.npy
- 验证集数据: {}/val/*.npy
- 测试集数据: {}/test/*.npy
- 数据标准化器: {}/data_scaler.pkl
- 处理汇总信息: {}/processing_summary.txt
- Scaler使用示例: {}/scaler_usage_example.py
- 数据加载辅助函数: {}/data_loader.py

下一步建议:
1. 使用data_loader.py加载数据集进行模型训练
2. 训练模型时记录使用的Scaler版本和处理方法
3. 对新数据集使用相同的Scaler进行标准化，确保模型泛化性
4. 考虑对处理后的数据进行可视化分析，验证标准化效果
""".format(
    output_base_dir, output_base_dir, output_base_dir, 
    output_base_dir, output_base_dir, output_base_dir, output_base_dir
))